# Semana 1 · Orientarse en la shell

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svillegasv/curso-genomica-filogenetica/blob/main/notebooks/semana-01/01-orientarse.ipynb)

**Genómica y Filogenética** · Semana 1 (U0 · Bootcamp UNIX) · Simón Villegas-Velásquez

**🎯 Resultado de aprendizaje:** RA1 · Al final de la sesión, usted se ubica en el sistema de archivos de Colab con rutas absolutas y relativas, crea la estructura de carpetas de un proyecto y usa los mensajes de error para corregir sus comandos.<br>
**⏱️ Duración:** ≈ 1,5 h · **🧰 Requisitos:** Una cuenta de Google. No hace falta experiencia previa con la terminal.

---

## 🧭 Dónde estamos
Semana 1 de 16. Este semestre van a comparar genomas de bacterias que llevan más de 35 años evolucionando en un
laboratorio (el LTEE). Esos datos no se abren con doble clic: son archivos de millones de líneas que se manejan con
la **shell**. Las semanas 1 a 3 son el *bootcamp*: hoy aprenden a orientarse, en la semana 2 a leer datos biológicos
y en la 3 a automatizar.

In [ ]:
#@title ⚙️ Configuración (ejecute esta celda primero) { display-mode: "form" }
import os, sys, urllib.request
from pathlib import Path

RAW = "https://raw.githubusercontent.com/svillegasv/curso-genomica-filogenetica/main"
_utils = next((p / "utils" for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "utils" / "curso.py").exists()), None)
if _utils is None:  # en Colab: se descargan las utilidades del curso
    _utils = Path(".curso/utils")
    _utils.mkdir(parents=True, exist_ok=True)
    for _f in ("curso.py", "estilo.py"):
        urllib.request.urlretrieve(f"{RAW}/utils/{_f}", _utils / _f)
sys.path.insert(0, str(_utils))

import curso
import estilo as es
import matplotlib.pyplot as plt
es.set_style()
print("Entorno listo ✔ | Colab:", curso.IN_COLAB)

In [ ]:
es.mapa_curso("unix");

## ❓ La pregunta biológica
El genoma del ancestro del LTEE (la cepa REL606) tiene unos 4,6 millones de letras, y una sola secuenciación de un
clon produce millones de lecturas. Ningún humano lee eso a ojo. **¿Cómo se organiza, se encuentra y se maneja ese
volumen de información sin abrir los archivos?** Hoy damos el primer paso: saber *dónde* estamos y *dónde* guardar
cada cosa.

## 📦 La carpeta heredada
Un estudiante del laboratorio se graduó y le dejó a usted su carpeta de trabajo. Tiene lo necesario para empezar…
pero está desordenada. Ejecute la celda siguiente para recibirla (no hace falta mirar su código).

> Si en algún momento quiere empezar de cero, vuelva a ejecutar esta celda: recrea `herencia/` y **borra** la carpeta
> `proyecto/` que haya creado.

In [ ]:
#@title 📦 Recibir la carpeta heredada { display-mode: "form" }
import shutil, tempfile

if curso.IN_COLAB:
    BASE = Path("/content")
else:  # pruebas automáticas o copia local
    BASE = Path(os.environ.get("CURSO_PROYECTO") or tempfile.mkdtemp(prefix="semana01-"))

for carpeta in ("herencia", "proyecto"):
    shutil.rmtree(BASE / carpeta, ignore_errors=True)
h = BASE / "herencia"
(h / "viejo").mkdir(parents=True)

(h / "REL606 genoma.fasta").write_text(
    ">NC_012967.1 Escherichia coli B str. REL606 (solo las primeras 240 letras de 4 629 812)\n"
    "AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTCTGATAGCAGC\n"
    "TTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTTATTGACTTAGGTCACTAAATACTTTAACCAA\n"
    "TATAGGCATAGCGCACAGACAGATAAAAATTACAGAGTACACAACATCCATGAAACGCATTAGCACCACC\n"
    "ATTACCACCACCATCACCATTACCACAGGT\n")
(h / "lecturas_clon1.fastq").write_text(
    "@lectura_1\nAGCTTTTCATTCTGACTGCAACGGGCAATA\n+\nIIIIIIIIIIIIIIIIIIIIIIIIIHHHGG\n"
    "@lectura_2\nTTCTGAACTGGTTACCTGCCGTGAGTAAAT\n+\nIIIIIIIIIIIIIIIIIIIIIIHHHGGFF#\n"
    "@lectura_3\nTATAGGCATAGCGCACAGACAGATAAAAAT\n+\nIIIIIIIIIIIIIIIIIIIIIIIIIIIIHH\n")
(h / "notas Juan.txt").write_text(
    "- El genoma del ancestro (REL606) está en esta carpeta.\n"
    "- Las lecturas son del clon 1.\n"
    "- No borrar nada de viejo/ (¿o sí?).\n")
(h / "analisis_FINAL_final2.sh").write_text(
    "#!/bin/bash\n"
    "# Cuenta cuántas lecturas tiene un archivo FASTQ (cada lectura ocupa 4 líneas)\n"
    "echo $(( $(wc -l < lecturas_clon1.fastq) / 4 ))\n")
(h / "viejo" / "resultados.csv").write_text("archivo,lecturas\nlecturas_clon1.fastq,3\n")
(h / ".pendientes.txt").write_text(
    "- Organizar esta carpeta (perdón).\n"
    "- Pregunta sin responder: ¿por qué algunos clones de Ara-3 acumulan muchas más mutaciones que otros?\n")

os.chdir(BASE)  # el cuaderno queda parado en la carpeta de trabajo
print("✔ Carpeta recibida en", h)

## Bloque 1 · Colab es un computador Linux
*≈ 20 min · L0.1*

Detrás de este cuaderno hay un computador real con Linux que Google le presta en algún lugar del mundo. Las celdas
normales le hablan a Python. Para darle órdenes al sistema operativo se usa la **shell** (el programa que interpreta
comandos; aquí se llama bash). En Colab hay dos formas de usarla: `!comando` para una línea y `%%bash` para una celda
completa.

### Ejemplo resuelto
> 🤔 **Antes de ejecutar, prediga:** ¿cómo se llama usted en este computador? ¿En qué carpeta está?

In [ ]:
%%bash
whoami    # ¿quién soy?
pwd       # ¿dónde estoy? (print working directory)
ls        # ¿qué hay aquí?

> 🔎 **Qué observamos.** Usted es `root`, el administrador: puede hacer cualquier cosa, incluso borrar lo que no
> debe. Está en `/content`, la carpeta de trabajo de Colab. Ahí está `herencia`, la carpeta que acaba de recibir.

### 🩺 Lea el error, aunque no haya mensaje
> 🤔 **Prediga:** después de estas dos celdas, ¿qué imprime `pwd`?

In [ ]:
!cd herencia

In [ ]:
!pwd

> 🔎 Imprime `/content`: el `cd` no se quedó, y nada lo avisó. Es el tipo de error más peligroso, el silencioso.
> Cada `!` abre una shell nueva (una *subshell*), hace su trabajo y se cierra, y el `cd` desaparece con ella. Para
> moverse de verdad se usa `%cd`, que cambia la carpeta del cuaderno mismo.

### Ejercicio guiado
Reemplace `COMPLETE` por su respuesta.

In [ ]:
# Ejercicio 1: entre a herencia de forma que el cambio se quede
# (versión del estudiante: %cd COMPLETE)
%cd herencia

In [ ]:
%%bash
pwd    # ¿y ahora? ¿Por qué esta vez la shell sí "ve" el cambio?

In [ ]:
#@title 🔑 Solución — Ejercicio 1 { display-mode: "form" }
# %cd herencia
# La shell nueva HEREDA la carpeta del cuaderno que la abre, igual que un hijo hereda la dirección de su casa.
# !cd cambia la carpeta del hijo (que muere enseguida); %cd cambia la del padre (el cuaderno).

## Bloque 2 · ¿Dónde estoy? Rutas
*≈ 25 min · L0.2*

```
/                       ← la raíz: todo cuelga de aquí
├── root/               ← su "casa" (~)
└── content/            ← la carpeta de trabajo de Colab
    ├── sample_data/
    └── herencia/       ← usted está aquí
        ├── REL606 genoma.fasta
        ├── lecturas_clon1.fastq
        ├── …
        └── viejo/
            └── resultados.csv
```

### Ejemplo resuelto 1: mirar con más detalle
> 🤔 **Antes de ejecutar, prediga:** ¿cuántos archivos hay en `herencia`?

In [ ]:
%%bash
ls        # lo básico
echo "---"
ls -F     # la / marca las carpetas
echo "---"
ls -a     # también los ocultos (empiezan con punto)

> 🔎 **Qué observamos.** `ls -a` revela `.pendientes.txt`, que antes no se veía, y además dos entradas raras: `.`
> (esta carpeta) y `..` (la de arriba). No son trucos de la pantalla: son nombres reales que se usan en las rutas.

### Rutas absolutas y relativas
Una ruta **absoluta** es como una dirección completa («Calle 10 # 5-20, Medellín»): empieza en `/` y funciona desde
cualquier lugar. Una **relativa** es como «dos cuadras a la izquierda»: depende de dónde esté parado.

### Ejemplo resuelto 2

In [ ]:
%%bash
ls viejo     # relativa: viejo/ está aquí adentro
echo "---"
ls ..        # relativa: la carpeta de arriba (/content)

In [ ]:
!ls /content/herencia/viejo     # absoluta: da lo mismo dónde esté

### 🩺 Lea el error
> 🤔 **Prediga:** ¿qué pasará?

In [ ]:
%cd viejo
!ls lecturas_clon1.fastq

> 🔎 `ls: cannot access 'lecturas_clon1.fastq': No such file or directory`. Léalo por partes: **quién** habla (`ls`),
> **qué** intentó (acceder), **a qué** (el archivo) y **por qué** falló (no existe). Pero el archivo sí existe… solo
> que no *aquí*. Casi siempre que vea este error, la primera pregunta es: ¿dónde estoy? (`pwd`).

### Ejercicio guiado
Sin moverse de `viejo/`:

In [ ]:
%%bash
# Ejercicio 2: liste el archivo de lecturas con una ruta RELATIVA
# (versión del estudiante: ls COMPLETE/lecturas_clon1.fastq)
ls ../lecturas_clon1.fastq

In [ ]:
# Ejercicio 3: ahora con una ruta ABSOLUTA
# (versión del estudiante: !ls COMPLETE)
!ls /content/herencia/lecturas_clon1.fastq

In [ ]:
#@title 🔑 Solución — Ejercicios 2 y 3 { display-mode: "form" }
# Ejercicio 2: ls ../lecturas_clon1.fastq      (.. = la carpeta de arriba, herencia/)
# Ejercicio 3: !ls /content/herencia/lecturas_clon1.fastq

> 🤔 **Ejercicio 4. Prediga** a qué carpeta llega este comando y luego ejecútelo.

In [ ]:
%cd ../..

In [ ]:
#@title 🔑 Solución — Ejercicio 4 { display-mode: "form" }
# Llega a /content: dos niveles arriba de viejo/ (viejo → herencia → content). Ahí empieza el bloque 3.

## Bloque 3 · Armar el proyecto
*≈ 30 min · L0.3 y L0.4*

La carpeta que heredó funciona, pero nadie más podría usarla. Todos los proyectos del curso tienen la misma forma:
`data/raw` (datos crudos, que nunca se modifican), `data/processed`, `results` y `scripts`. Si un proyecto sigue esa
forma, cualquiera sabe dónde buscar.

### Ejemplo resuelto 1: crear
> 🤔 **Antes de ejecutar, prediga:** ¿para qué sirve la `-p`?

In [ ]:
%%bash
mkdir -p proyecto/data/raw proyecto/data/processed proyecto/results proyecto/scripts
touch proyecto/README.md     # crea un archivo vacío
ls -F proyecto proyecto/data

> 🔎 **Qué observamos.** `-p` crea las carpetas intermedias (`data/`) que hagan falta. Sin ella, `mkdir` se queja de
> que `proyecto/data` no existe.

### Ejemplo resuelto 2: copiar frente a mover
> 🤔 **Prediga:** después de esta celda, ¿qué queda en `herencia/`?

In [ ]:
%%bash
cp herencia/lecturas_clon1.fastq proyecto/data/raw/        # copia
mv herencia/viejo/resultados.csv proyecto/results/         # mueve
ls herencia herencia/viejo

> 🔎 **Qué observamos.** `cp` deja el original y `mv` se lo lleva. Regla del curso: los datos crudos se **copian**,
> nunca se editan ni se mueven sin respaldo.

### Nombres que no dan problemas
Sin espacios ni tildes, en minúsculas o con `_`, y descriptivos. Nunca `final_final2`.

> 🤔 **Prediga** qué pasa con esta línea:

In [ ]:
!mv herencia/REL606 genoma.fasta proyecto/data/raw/

> 🔎 La shell separa las palabras por espacios, así que entiende que hay *dos* archivos: `herencia/REL606` y
> `genoma.fasta`. Ninguno existe. Un espacio en un nombre es un error que espera su turno. Las comillas lo arreglan:
> `"herencia/REL606 genoma.fasta"`.

### 🩺 Lea el error: pedir ayuda

In [ ]:
!man mv

> 🔎 En Colab responde *"This system has been minimized…"*: el manual no está instalado para ahorrar espacio. Hay
> otras dos salidas: `mv --help`, que es corta y viene con el comando, y las páginas completas en
> [man7.org](https://man7.org/linux/man-pages/).

In [ ]:
!mv --help

### Ejercicio guiado

In [ ]:
%%bash
# Ejercicio 5: mueva el genoma a data/raw y póngale, de paso, un nombre sin espacios
# (versión del estudiante: mv "herencia/REL606 genoma.fasta" proyecto/data/raw/COMPLETE)
mv "herencia/REL606 genoma.fasta" proyecto/data/raw/REL606_genoma.fasta

In [ ]:
%%bash
# Ejercicio 6: el script va a scripts/, con un nombre que diga qué hace
# (para saber qué hace, ábralo con doble clic en el panel de archivos 📁 de la izquierda)
# (versión del estudiante: mv herencia/analisis_FINAL_final2.sh proyecto/scripts/COMPLETE)
mv herencia/analisis_FINAL_final2.sh proyecto/scripts/contar_lecturas.sh

In [ ]:
%%bash
# Ejercicio 7: 🤔 prediga a dónde va lo que se borra. Luego busque en `rm --help` qué opción borra carpetas
# y elimine viejo/
# (versión del estudiante: rm COMPLETE herencia/viejo)
rm -r herencia/viejo

In [ ]:
#@title 🔑 Solución — Ejercicios 5, 6 y 7 { display-mode: "form" }
# Ejercicio 5: mv "herencia/REL606 genoma.fasta" proyecto/data/raw/REL606_genoma.fasta
# Ejercicio 6: por ejemplo contar_lecturas.sh: el script cuenta las lecturas de un FASTQ
# Ejercicio 7: rm -r herencia/viejo
#   Lo borrado NO va a ninguna papelera: desaparece. Por eso los datos crudos tienen copia.

### ✔ Verificación
Ejecute la celda: revisa que su proyecto tenga la forma esperada.

In [ ]:
#@title ✔ ¿Quedó bien el proyecto? { display-mode: "form" }
revisar = [
    (BASE / "proyecto/data/raw/REL606_genoma.fasta", "Ejercicio 5: ¿movió el genoma con el nombre nuevo?"),
    (BASE / "proyecto/data/raw/lecturas_clon1.fastq", "Ejemplo 2: ¿copió las lecturas a data/raw?"),
    (BASE / "herencia/lecturas_clon1.fastq", "Las lecturas originales se COPIAN, no se mueven."),
    (BASE / "proyecto/data/processed", "Ejemplo 1: falta data/processed."),
    (BASE / "proyecto/results/resultados.csv", "Ejemplo 2: ¿movió resultados.csv a results/?"),
    (BASE / "proyecto/README.md", "Ejemplo 1: falta el README.md (touch)."),
]
for ruta, pista in revisar:
    print(("✔ " if ruta.exists() else "✘ ") + str(ruta.relative_to(BASE)) + ("" if ruta.exists() else "  → " + pista))
scripts = list((BASE / "proyecto/scripts").glob("*.sh"))
print(("✔ proyecto/scripts/" + scripts[0].name) if scripts else "✘ proyecto/scripts/  → Ejercicio 6: ¿movió el script?")
print("✘ herencia/viejo  → Ejercicio 7: aún existe" if (BASE / "herencia/viejo").exists() else "✔ herencia/viejo ya no existe")

## ✅ Compruebe su comprensión
1. Usted ejecuta `!cd results` y en la celda siguiente `!ls`. ¿Qué carpeta se lista y por qué?
2. Está en `/content/proyecto/data/raw`. Escriba dos rutas hacia `scripts/`: una relativa y una absoluta.
3. ¿Qué diferencia hay entre `cp` y `mv`? ¿Por qué los datos crudos se copian y no se mueven?
4. ¿Qué hace la shell con `ls notas Juan.txt`? ¿Cómo lo arregla?

In [ ]:
#@title 🔑 Respuestas { display-mode: "form" }
# 1. La carpeta del cuaderno: el cd murió con su subshell.
# 2. Relativa: ../../scripts   ·   Absoluta: /content/proyecto/scripts
# 3. cp deja el original; mv se lo lleva. Si algo sale mal, el dato crudo sigue intacto.
# 4. Busca dos archivos, "notas" y "Juan.txt". Se arregla con comillas: ls "notas Juan.txt"

## 🚀 Para ir más allá (opcional)
- En `herencia/` quedan `notas Juan.txt` y `.pendientes.txt`. Decida dónde van en el proyecto y con qué nombre, y
  muévalos.
- Abra `proyecto/README.md` desde el panel de archivos de Colab (📁, doble clic) y escriba en tres líneas qué contiene
  cada carpeta.
- Lea `.pendientes.txt`: el estudiante anterior dejó una pregunta sin responder. Es la pregunta del semestre, y llega
  completa en la semana 4.

## 📚 Referencias
- Software Carpentry. *The Unix Shell*. https://swcarpentry.github.io/shell-novice/
- Data Carpentry. *Introduction to the Command Line for Genomics*. https://datacarpentry.github.io/shell-genomics/
- Páginas del manual de Linux. https://man7.org/linux/man-pages/
- Noble, W. S. (2009). A quick guide to organizing computational biology projects. *PLoS Computational Biology*,
  5(7), e1000424. https://doi.org/10.1371/journal.pcbi.1000424
- Wilson, G., Bryan, J., Cranston, K., Kitzes, J., Nederbragt, L. y Teal, T. K. (2017). Good enough practices in
  scientific computing. *PLoS Computational Biology*, 13(6), e1005510. https://doi.org/10.1371/journal.pcbi.1005510
- Jeong, H., Barbe, V., Lee, C. H., et al. (2009). Genome sequences of *Escherichia coli* B strains REL606 and
  BL21(DE3). *Journal of Molecular Biology*, 394(4), 644–652. https://doi.org/10.1016/j.jmb.2009.09.052
- Tenaillon, O., Barrick, J. E., Ribeck, N., et al. (2016). Tempo and mode of genome evolution in a 50,000-generation
  experiment. *Nature*, 536, 165–170. https://doi.org/10.1038/nature18959

---

<sub>**Genómica y Filogenética** · © 2026 Simón Villegas-Velásquez · Código bajo licencia MIT · Texto y figuras bajo CC BY 4.0 ·
[Repositorio](https://github.com/svillegasv/curso-genomica-filogenetica)</sub>